### Reranking Hybrid Search Statergies

Re-ranking is a second-stage filtering process in retrieval systems, especially in RAG pipelines, where we:

1. First use a fast retriever (like BM25, FAISS, hybrid) to fetch top-k documents quickly.

2. Then use a more accurate but slower model (like a cross-encoder or LLM) to re-score and reorder those documents by relevance to the query.

👉 It ensures that the most relevant documents appear at the top, improving the final answer from the LLM.

In [9]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

In [10]:
## load text file
loader=TextLoader("langchain_sample.txt")
raw_docs=loader.load()

# Split text into document chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(raw_docs)
docs


[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.\nBM25 is a traditional 

In [11]:
## user query
query="How can i use langchain to build an application with memory and tools?"

In [12]:
### FAISS and Huggingface model Embeddings

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(docs,embedding_model)
retriever=vectorstore.as_retriever(search_kwargs={"k":8})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
print(vectorstore)

print("\n--- Index ---")
print("Type:", type(vectorstore.index).__name__)
print("Total vectors:", vectorstore.index.ntotal)
print("Vector dimension:", vectorstore.index.d)

print("\n--- Docstore ---")
print("Documents stored:", len(vectorstore.docstore._dict))

print("\n--- Index position -> document id ---")
for i, doc_id in vectorstore.index_to_docstore_id.items():
    print(i, doc_id)

print("\n--- Vectors and documents ---")
vectors = vectorstore.index.reconstruct_n(0, vectorstore.index.ntotal)
for i, doc_id in vectorstore.index_to_docstore_id.items():
    doc = vectorstore.docstore.search(doc_id)
    print(f"\n[{i}] id={doc_id}")
    print("metadata:", doc.metadata)
    print(f"vector (first 5 of {len(vectors[i])}):", vectors[i][:5])
    print("text:", doc.page_content[:100], "...")



--- Index ---
Type: IndexFlatL2
Total vectors: 6
Vector dimension: 384

--- Docstore ---
Documents stored: 6

--- Index position -> document id ---
0 47e7cc3b-939e-43d4-92db-8ba4fcd3a5cd
1 e0da0a6c-bf1f-4617-83de-5c4fdc7e75cb
2 5adc5a72-ee03-4aea-aa6f-5f3cfefba0e2
3 4c5425fd-96bd-4d27-8569-e242f4085eaa
4 92238c2f-3b83-4171-8e29-5b65ef118ccd
5 7a7cb2ac-d103-4dad-b584-e0e0a15254ab

--- Vectors and documents ---

[0] id=47e7cc3b-939e-43d4-92db-8ba4fcd3a5cd
metadata: {'source': 'langchain_sample.txt'}
vector (first 5 of 384): [ 0.00157215 -0.04676702  0.03153919 -0.07234115  0.00547087]
text: LangChain is a flexible framework designed for developing applications powered by large language mod ...

[1] id=e0da0a6c-bf1f-4617-83de-5c4fdc7e75cb
metadata: {'source': 'langchain_sample.txt'}
vector (first 5 of 384): [-0.04351314 -0.02569422  0.0204226  -0.02410712  0.01556136]
text: LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This e ...

[2] id=5a

In [14]:
## OpenAI Embedding
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
from langchain_openai import OpenAIEmbeddings

embeddings=OpenAIEmbeddings()
vectorstore_openai=FAISS.from_documents(docs,embeddings)
retriever_openai=vectorstore_openai.as_retriever(search_kwargs={"k":8})




In [15]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A42C2AD6D0>, search_kwargs={'k': 8})

In [16]:
retriever_openai

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A4683F7D90>, search_kwargs={'k': 8})

print(vectorstore)

print("\n--- Index ---")
print("Type:", type(vectorstore.index).__name__)
print("Total vectors:", vectorstore.index.ntotal)
print("Vector dimension:", vectorstore.index.d)

print("\n--- Docstore ---")
print("Documents stored:", len(vectorstore.docstore._dict))

print("\n--- Index position -> document id ---")
for i, doc_id in vectorstore.index_to_docstore_id.items():
    print(i, doc_id)

print("\n--- Vectors and documents ---")
vectors = vectorstore.index.reconstruct_n(0, vectorstore.index.ntotal)
for i, doc_id in vectorstore.index_to_docstore_id.items():
    doc = vectorstore.docstore.search(doc_id)
    print(f"\n[{i}] id={doc_id}")
    print("metadata:", doc.metadata)
    print(f"vector (first 5 of {len(vectors[i])}):", vectors[i][:5])
    print("text:", doc.page_content[:100], "...")

In [17]:
## prompt and use the llm
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm=init_chat_model("groq:gemma2-9b-it")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'Gemma 2 9B', 'status': 'deprecated', 'release_date': '2024-06-27', 'last_updated': '2024-06-27', 'open_weights': True, 'max_input_tokens': 8192, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A46871D6A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A46871E120>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [18]:
# Prompt Template
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)
""")

In [19]:
retrieved_docs=retriever.invoke(query)
retrieved_docs

[Document(id='7a7cb2ac-d103-4dad-b584-e0e0a15254ab', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='47e7cc3b-939e-43d4-92db-8ba4fcd3a5cd', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='e0da0a6c-bf1f-4617-83de-5c4fdc7e75cb', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging F

In [20]:
chain=prompt| llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n- Think about the relevance of each document to the user\'s question.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput format: comma-separated document indices (e.g., 2,1,3,0,...)\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'Gemma 2 9B', 'status': 'deprecated', 'release_date': '2024-06-27', 'last_updated': '2024-06-27', 'open_weights': True, 'max_input_tokens': 8192, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': F

In [21]:
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)

In [22]:
doc_lines

['1. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.',
 '2. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '3. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '4. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and comp

In [23]:
formatted_docs

'1. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.\n2. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.\n3. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.\n4. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed ind

In [24]:
response=chain.invoke({"question":query,"documents":formatted_docs})
response

BadRequestError: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

In [ ]:
# Step 5: Parse and rerank
indices = [int(x.strip()) - 1 for x in response.split(",") if x.strip().isdigit()]
indices

[1, 0, 3, 4]

In [ ]:
retrieved_docs

[Document(id='709da153-eaee-411b-bd17-b5fc443be7da', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='a217edf2-c10c-40ff-9317-37c4e3388096', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='ec8fd256-9859-4aa8-becb-b505799be510', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging F

In [ ]:
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i < len(retrieved_docs)]
reranked_docs

[Document(id='a217edf2-c10c-40ff-9317-37c4e3388096', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='709da153-eaee-411b-bd17-b5fc443be7da', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='d075ce19-78fc-4392-bde0-0a6a8a45a7ec', metadata={'source': 'langchain_sample.txt'}, page_content='FAISS is a popular library used for fast approximate nearest neighbor search 

In [ ]:
# Step 6: Show results
print("\n📊 Final Reranked Results:")
for i, doc in enumerate(reranked_docs, 1):
    print(f"\nRank {i}:\n{doc.page_content}")


📊 Final Reranked Results:

Rank 1:
LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.

Rank 2:
LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.

Rank 3:
FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed indexes, which makes it scalable for large document stores.
Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering w